In [4]:
import numpy as np
import pandas as pd

import hvplot.pandas  # adds the .hvplot accessor
import hvplot         # for hvplot.extension(...)
import panel as pn

hvplot.extension("bokeh")
pn.extension()

# --- Example "ICA-like" timeseries DataFrame: datetime index, components in columns ---
n = 2000
t = pd.date_range("2025-01-01", periods=n, freq="S")

df = pd.DataFrame(
    {
        "ICA_01": np.sin(np.linspace(0, 40, n)) + 0.10 * np.random.randn(n),
        "ICA_02": np.cos(np.linspace(0, 22, n)) + 0.10 * np.random.randn(n),
        "ICA_03": np.sin(np.linspace(0, 10, n) ** 1.1) + 0.10 * np.random.randn(n),
        "ICA_04": 0.4 * np.random.randn(n).cumsum(),
    },
    index=t,
)
df.index.name = "time"

df.head()

/tmp/ipykernel_884257/2849219838.py:13: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  t = pd.date_range("2025-01-01", periods=n, freq="S")


,ICA_01,ICA_02,ICA_03,ICA_04
time,,,,
2025-01-01 00:00:00,-0.007212,0.829599,-0.079851,0.633441
2025-01-01 00:00:01,-0.006769,0.939377,0.031583,0.062355
2025-01-01 00:00:02,0.026172,0.832043,-0.023920,0.599723
2025-01-01 00:00:03,0.240693,1.090811,-0.086357,0.855487
2025-01-01 00:00:04,0.125494,0.975426,-0.014692,1.048567


In [5]:
# --- Reshape to "long" format so groupby can work on a column ---
# long columns will be: time, component, value
long = (
    df.stack()               # stacks columns into rows
      .rename("value")       # name the stacked values
      .reset_index()         # turn index levels into columns
      .rename(columns={"level_1": "component"})
)

long.head(5)

,time,component,value
0,2025-01-01 00:00:00,ICA_01,-0.007212
1,2025-01-01 00:00:00,ICA_02,0.829599
2,2025-01-01 00:00:00,ICA_03,-0.079851
3,2025-01-01 00:00:00,ICA_04,0.633441
4,2025-01-01 00:00:01,ICA_01,-0.006769


In [ ]:
# --- Interactive plot: dropdown to choose the component ---
plot = long.hvplot.line(
    x="time",
    y="value",
    groupby="component",     # <- creates the interactive widget
    width=900,
    height=300,
    title="ICA component (choose from dropdown)",
    grid=True,
)

plot


In [3]:
df.columns

Index(['ICA_01', 'ICA_02', 'ICA_03', 'ICA_04'], dtype='object')